# Training semantic segmentation (DeepLabv3+ or PSPNet) in Google Colab with CVAT masks and Azure Blob storage

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/datasantana/sam-image-processing/blob/main/notebooks/semantic_segmentation_training.ipynb)

In [ ]:
# 🔧 Install Dependencies
# Run this cell first when opening the notebook in Google Colab

!pip install azure-storage-blob>=12.17.0
!pip install opencv-python>=4.8.0  
!pip install pillow>=10.0.0
!pip install numpy>=1.24.0
!pip install matplotlib>=3.7.0

print("🎉 Dependencies installed successfully!")

# 🛰️ Semantic Segmentation Training Setup

This notebook initializes the connection to Azure Blob Storage, downloads drone imagery and labeled masks, and prepares reproducible training and validation datasets for semantic segmentation using DeepLabv3+ or PSPNet.

**Dataset structure:**
- Images: `assets/drone-imagery`
- Masks: `assets/pascual_annotation_masks/SegmentationClass`
- Label map: `assets/pascual_annotation_masks/labelmap.txt`

**Azure Blob Storage:**
- Container: `general-purpose`
- Connection string: stored securely in `AZURE_CONN_STR`

## 1: Azure Blob Connection & Download

In [ ]:
# 📦 Connect to Azure Blob Storage and download images and masks
from azure.storage.blob import BlobServiceClient
import os

# 🔐 Azure Connection Configuration
# Option 1: Set as environment variable (recommended for Colab)
# import os
# AZURE_CONN_STR = os.getenv('AZURE_CONNECTION_STRING')

# Option 2: Use getpass for secure input (uncomment to use)
# from getpass import getpass
# AZURE_CONN_STR = getpass('Enter Azure Connection String: ')

# Option 3: Set directly (replace with your actual connection string)
AZURE_CONN_STR = "DefaultEndpointsProtocol=...<place complete string here>"

if AZURE_CONN_STR == "DefaultEndpointsProtocol=...<place complete string here>":
    print("⚠️  Please set your Azure connection string!")
    print("Uncomment one of the options above or set AZURE_CONN_STR variable")
    raise ValueError("Azure connection string not configured")

CONTAINER_NAME = "general-purpose"

# Blob Storage paths (where data is stored in Azure)
LOCAL_IMG_DIR = "assets/drone-imagery"
LOCAL_MASK_DIR = "assets/pascual_annotation_masks/SegmentationClass"
LOCAL_LABELMAP_DIR = "assets/pascual_annotation_masks"

# Local directories (where to download files)
LOCAL_DOWNLOAD_IMG_DIR = "assets/images"
LOCAL_DOWNLOAD_MASK_DIR = "assets/masks"

os.makedirs(LOCAL_DOWNLOAD_IMG_DIR, exist_ok=True)
os.makedirs(LOCAL_DOWNLOAD_MASK_DIR, exist_ok=True)

# Connect to blob container
try:
    blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONN_STR)
    container_client = blob_service_client.get_container_client(CONTAINER_NAME)
    print("✅ Successfully connected to Azure Blob Storage")
except Exception as e:
    print(f"❌ Failed to connect to Azure Blob Storage: {e}")
    raise

def download_blobs(prefix, local_dir, file_types=None):
    """
    Download blobs with optional file type filtering
    
    Args:
        prefix: Blob prefix to search for
        local_dir: Local directory to download to
        file_types: List of extensions (case-insensitive) or None to download all files
                   e.g., ['.jpg', '.jpeg', '.png'] or None
    """
    print(f"🔍 Looking for blobs with prefix '{prefix}'...")
    if file_types:
        print(f"   File types: {file_types} (case-insensitive)")
    else:
        print("   Downloading all file types")
    
    try:
        blobs = list(container_client.list_blobs(name_starts_with=prefix))
        print(f"📁 Found {len(blobs)} total blobs with prefix '{prefix}'")
        
        downloaded_count = 0
        for blob in blobs:
            print(f"  - Checking blob: {blob.name}")
            
            # Check file type if specified
            should_download = True
            if file_types:
                should_download = any(blob.name.lower().endswith(ext.lower()) for ext in file_types)
            
            if should_download:
                local_path = os.path.join(local_dir, os.path.basename(blob.name))
                if not os.path.exists(local_path):
                    print(f"    ⬇️ Downloading {blob.name} -> {local_path}")
                    try:
                        with open(local_path, "wb") as f:
                            blob_data = container_client.download_blob(blob.name).readall()
                            f.write(blob_data)
                        downloaded_count += 1
                        print(f"    ✅ Downloaded {blob.name} ({len(blob_data)} bytes)")
                    except Exception as e:
                        print(f"    ❌ Failed to download {blob.name}: {e}")
                else:
                    print(f"    ⏭️ Skipping {blob.name} (already exists)")
            else:
                print(f"    ⏭️ Skipping {blob.name} (file type not in {file_types})")
        
        print(f"📊 Downloaded {downloaded_count} files from {prefix} to {local_dir}")
        
    except Exception as e:
        print(f"❌ Error listing blobs with prefix '{prefix}': {e}")

# Download images and masks with flexible file type matching
# Images: Support common image formats (case-insensitive)
download_blobs(LOCAL_IMG_DIR, LOCAL_DOWNLOAD_IMG_DIR, ['.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.tif'])

# Masks: Support common mask formats (case-insensitive)  
download_blobs(LOCAL_MASK_DIR, LOCAL_DOWNLOAD_MASK_DIR, ['.png', '.jpg', '.jpeg', '.bmp'])

## 2: 🧪 Dataset Preparation

This section reads and sorts the downloaded images and masks, then creates a reproducible split:
- First 53 images → Training set
- Remaining images → Validation set

Each image is matched with its corresponding mask by filename.

In [ ]:
# List and sort all image and mask files (no format validation)
image_files = sorted([f for f in os.listdir(LOCAL_DOWNLOAD_IMG_DIR)])
mask_files = sorted([f for f in os.listdir(LOCAL_DOWNLOAD_MASK_DIR)])

print(f"📁 Found {len(image_files)} image files")
print(f"📁 Found {len(mask_files)} mask files")

# Match image-mask pairs by base filename (remove extensions)
def get_base_name(filename):
    return os.path.splitext(filename)[0]

# Create pairs based on matching base names
image_base_names = {get_base_name(f): f for f in image_files}
mask_base_names = {get_base_name(f): f for f in mask_files}

# Find matching pairs
matching_base_names = set(image_base_names.keys()) & set(mask_base_names.keys())
pairs = [(image_base_names[base], mask_base_names[base]) for base in sorted(matching_base_names)]

print(f"🔗 Found {len(pairs)} matching image-mask pairs")

# Reproducible split: first 53 for training
train_pairs = pairs[:53]
val_pairs = pairs[53:]

# Build full path dictionaries
def build_dataset(pairs, img_dir, mask_dir):
    return [
        {
            "img_path": os.path.join(img_dir, img),
            "seg_map_path": os.path.join(mask_dir, mask)
        }
        for img, mask in pairs
    ]

train_dataset = build_dataset(train_pairs, LOCAL_DOWNLOAD_IMG_DIR, LOCAL_DOWNLOAD_MASK_DIR)
val_dataset = build_dataset(val_pairs, LOCAL_DOWNLOAD_IMG_DIR, LOCAL_DOWNLOAD_MASK_DIR)

print(f"✅ Train set: {len(train_dataset)} images")
print(f"✅ Val set: {len(val_dataset)} images")
print(f"📁 Images directory: {LOCAL_DOWNLOAD_IMG_DIR}")
print(f"📁 Masks directory: {LOCAL_DOWNLOAD_MASK_DIR}")
print(f"🔗 Blob source - Images: {LOCAL_IMG_DIR}")
print(f"🔗 Blob source - Masks: {LOCAL_MASK_DIR}")

## 3: 🚀 Model Training Setup

Now we'll set up the semantic segmentation training pipeline with support for:
- **DeepLabv3+** with ResNet backbone
- **PSPNet** with ResNet backbone  
- Easy model switching
- Configurable training parameters
- Label map integration

### 3.1: 🔧 Deep Learning Environment Setup

In [ ]:
# 🔧 Install Deep Learning Dependencies
# PyTorch and related packages
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install segmentation-models-pytorch>=0.3.3
!pip install albumentations>=1.3.0
!pip install timm>=0.9.0

# Additional ML utilities
!pip install scikit-learn>=1.3.0
!pip install tqdm>=4.65.0
!pip install tensorboard>=2.13.0

print("🎉 Deep learning environment setup complete!")

### 3.2: 📋 Model Configuration

In [ ]:
# 📋 Model and Training Configuration
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2

# 🎯 Model Selection - Change this to switch models
MODEL_ARCH = "DeepLabv3+"  # Options: "DeepLabv3+", "PSPNet"
BACKBONE = "resnet50"      # Options: "resnet18", "resnet34", "resnet50", "resnet101", "resnet152"

# 🏋️ Training Configuration
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
NUM_EPOCHS = 50
IMAGE_SIZE = 512
NUM_WORKERS = 2

# 📊 Model Architecture Settings
if MODEL_ARCH == "DeepLabv3+":
    print(f"🎯 Using DeepLabv3+ with {BACKBONE} backbone")
    model_class = smp.DeepLabV3Plus
elif MODEL_ARCH == "PSPNet":
    print(f"🎯 Using PSPNet with {BACKBONE} backbone")
    model_class = smp.PSPNet
else:
    raise ValueError(f"Unsupported model architecture: {MODEL_ARCH}")

# 🖥️ Device Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🖥️ Using device: {device}")

# 📁 Output Configuration
MODEL_SAVE_DIR = "models"
LOGS_DIR = "logs"
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

print(f"✅ Configuration complete!")
print(f"   Model: {MODEL_ARCH}")
print(f"   Backbone: {BACKBONE}")
print(f"   Batch Size: {BATCH_SIZE}")
print(f"   Learning Rate: {LEARNING_RATE}")
print(f"   Image Size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"   Epochs: {NUM_EPOCHS}")

### 3.3: 🏷️ Class Labels and Label Map

In [ ]:
# 🏷️ Load and Process Label Map
import numpy as np
from PIL import Image
import json

# Local path where labelmap will be downloaded
local_labelmap_path = "assets/labelmap.txt"

try:
    # Try to download labelmap from blob storage
    print("🔍 Looking for labelmap in blob storage...")
    labelmap_blobs = list(container_client.list_blobs(name_starts_with=LOCAL_LABELMAP_DIR))
    
    labelmap_found = False
    for blob in labelmap_blobs:
        if 'labelmap' in blob.name.lower() and blob.name.endswith('.txt'):
            print(f"📥 Downloading labelmap: {blob.name} -> {local_labelmap_path}")
            with open(local_labelmap_path, "wb") as f:
                f.write(container_client.download_blob(blob.name).readall())
            labelmap_found = True
            break
    
    if not labelmap_found:
        raise FileNotFoundError("No labelmap.txt file found in blob storage")
    
    # Load labelmap from downloaded file
    if os.path.exists(local_labelmap_path):
        with open(local_labelmap_path, 'r') as f:
            class_names = [line.strip() for line in f.readlines() if line.strip()]
        print(f"✅ Loaded {len(class_names)} classes from labelmap")
    else:
        raise FileNotFoundError("Labelmap download failed")
        
except Exception as e:
    print(f"⚠️ Could not load labelmap: {e}")
    print("🔧 Creating default labelmap by analyzing mask files...")
    
    # Analyze a few mask files to determine unique classes
    sample_masks = mask_files[:min(10, len(mask_files))]
    unique_values = set()
    
    for mask_file in sample_masks:
        mask_path = os.path.join(LOCAL_DOWNLOAD_MASK_DIR, mask_file)
        try:
            mask = np.array(Image.open(mask_path))
            unique_values.update(np.unique(mask))
        except Exception as mask_error:
            print(f"⚠️ Could not read mask {mask_file}: {mask_error}")
    
    # Create class names
    unique_values = sorted(list(unique_values))
    class_names = [f"class_{i}" for i in unique_values]
    
    print(f"🔍 Found {len(unique_values)} unique classes: {unique_values}")

# Set up classes
NUM_CLASSES = len(class_names)
print(f"📊 Number of classes: {NUM_CLASSES}")
print(f"🏷️ Class names: {class_names}")

# Create class to index mapping
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
idx_to_class = {idx: name for idx, name in enumerate(class_names)}

print(f"✅ Label mapping complete!")
print(f"   Classes: {NUM_CLASSES}")
print(f"   Mapping: {dict(list(class_to_idx.items())[:5])}{'...' if len(class_to_idx) > 5 else ''}")
print(f"🔗 Blob source - Labelmap: {LOCAL_LABELMAP_DIR}")
print(f"📁 Local labelmap: {local_labelmap_path if os.path.exists(local_labelmap_path) else 'Generated from masks'}")

### 3.4: 🔄 Data Transforms and Dataset

In [ ]:
# 🔄 Data Transforms and Custom Dataset
from torch.utils.data import Dataset
import cv2

class SegmentationDataset(Dataset):
    def __init__(self, dataset, transforms=None):
        self.dataset = dataset
        self.transforms = transforms
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        item = self.dataset[idx]
        
        # Load image and mask
        image = cv2.imread(item['img_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(item['seg_map_path'], cv2.IMREAD_GRAYSCALE)
        
        # Apply transforms
        if self.transforms:
            transformed = self.transforms(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']
        
        return image, mask.long()

# 🎨 Define Transforms
train_transforms = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.RandomGamma(p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transforms = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

# 📊 Create Datasets
train_ds = SegmentationDataset(train_dataset, transforms=train_transforms)
val_ds = SegmentationDataset(val_dataset, transforms=val_transforms)

# 🔄 Create DataLoaders
train_loader = DataLoader(
    train_ds, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"✅ Datasets created successfully!")
print(f"   Training batches: {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")
print(f"   Samples per batch: {BATCH_SIZE}")

# 🧪 Test data loading
try:
    sample_image, sample_mask = next(iter(train_loader))
    print(f"📊 Data loading test successful!")
    print(f"   Image shape: {sample_image.shape}")
    print(f"   Mask shape: {sample_mask.shape}")
    print(f"   Image range: [{sample_image.min():.3f}, {sample_image.max():.3f}]")
    print(f"   Mask classes: {torch.unique(sample_mask)}")
except Exception as e:
    print(f"❌ Data loading test failed: {e}")

### 3.5: 🏗️ Model Initialization

In [ ]:
# 🏗️ Initialize Model
model = model_class(
    encoder_name=BACKBONE,
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES,
)

model = model.to(device)

# 📊 Model Summary
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

total_params = count_parameters(model)
print(f"🏗️ Model initialized: {MODEL_ARCH}")
print(f"   Backbone: {BACKBONE}")
print(f"   Total parameters: {total_params:,}")
print(f"   Classes: {NUM_CLASSES}")

# 🎯 Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=1e-4
)

# 📈 Learning Rate Scheduler (removed deprecated 'verbose' parameter)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=5
)

print(f"✅ Training setup complete!")
print(f"   Loss function: CrossEntropyLoss")
print(f"   Optimizer: AdamW (lr={LEARNING_RATE})")
print(f"   Scheduler: ReduceLROnPlateau")

# 🧪 Test forward pass
try:
    with torch.no_grad():
        dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
        dummy_output = model(dummy_input)
        print(f"🧪 Forward pass test successful!")
        print(f"   Input shape: {dummy_input.shape}")
        print(f"   Output shape: {dummy_output.shape}")
except Exception as e:
    print(f"❌ Forward pass test failed: {e}")

### 3.6: 🧪 Forward Pass Test Fix

The error "Expected more than 1 value per channel when training" occurs because:

- **BatchNorm layers** in the model expect batch size > 1 when in training mode
- The test uses batch size = 1, which causes BatchNorm to fail
- **Solution**: Set model to `eval()` mode for testing, then back to `train()` mode

In [ ]:
# 🧪 Fixed Forward Pass Test
# The BatchNorm error occurs because the model is in training mode with batch_size=1
# Solution: Set model to eval() mode for testing

print("🧪 Testing model forward pass...")

try:
    # Set model to evaluation mode to avoid BatchNorm issues
    model.eval()
    
    with torch.no_grad():
        # Test with different batch sizes
        test_batch_sizes = [1, 2]
        
        for batch_size in test_batch_sizes:
            dummy_input = torch.randn(batch_size, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
            dummy_output = model(dummy_input)
            
            print(f"✅ Batch size {batch_size} test successful!")
            print(f"   Input shape: {dummy_input.shape}")
            print(f"   Output shape: {dummy_output.shape}")
            print(f"   Output classes: {dummy_output.shape[1]} (expected: {NUM_CLASSES})")
    
    # Set back to training mode for actual training
    model.train()
    print(f"🔄 Model set back to training mode")
    
except Exception as e:
    print(f"❌ Forward pass test failed: {e}")
    print("💡 This might indicate a model architecture issue")

print(f"\n🎯 Model is ready for training!")
print(f"   Architecture: {MODEL_ARCH}")
print(f"   Backbone: {BACKBONE}")
print(f"   Classes: {NUM_CLASSES}")
print(f"   Parameters: {count_parameters(model):,}")

## 4: 🚀 Training Loop

**Current Status**: We have a **READY-TO-TRAIN** model, not a trained one yet!

So far we've only:
- ✅ Downloaded data from Azure Blob Storage
- ✅ Prepared datasets and data loaders  
- ✅ Initialized the model architecture
- ✅ Set up loss function, optimizer, and scheduler

**Next**: Run the actual training to get a **TRAINED** model!

### 🔧 Mask Value Fix - Critical!

In [ ]:
# 🚨 CRITICAL FIX: Mask values must be in range [0, NUM_CLASSES-1]
# IMPORTANT: This preserves the original class names from labelmap!

print("🔍 Analyzing mask values...")
print(f"📋 Current class setup from labelmap:")
print(f"   Classes: {NUM_CLASSES}")
print(f"   Class names: {class_names}")
print(f"   Expected range: [0, {NUM_CLASSES-1}]")

# Check actual mask values in a sample of files
sample_mask_values = set()
problematic_files = []

for i, mask_file in enumerate(mask_files[:10]):  # Check first 10 masks
    mask_path = os.path.join(LOCAL_DOWNLOAD_MASK_DIR, mask_file)
    try:
        mask = np.array(Image.open(mask_path))
        unique_vals = np.unique(mask)
        sample_mask_values.update(unique_vals)
        
        # Check for values outside expected range
        max_expected = NUM_CLASSES - 1
        if np.max(unique_vals) > max_expected:
            problematic_files.append((mask_file, unique_vals))
        
        print(f"  {mask_file}: {unique_vals}")
        
    except Exception as e:
        print(f"  ❌ Error reading {mask_file}: {e}")

print(f"\n📊 Analysis Results:")
print(f"   All unique values found in masks: {sorted(sample_mask_values)}")
print(f"   Expected range for model: [0, {NUM_CLASSES-1}]")
print(f"   Problematic files: {len(problematic_files)}")

if problematic_files:
    print(f"\n⚠️ PROBLEM DETECTED!")
    print(f"   Mask values go up to {max(sample_mask_values)}, but model expects max {NUM_CLASSES-1}")
    print(f"   This will cause 'IndexError: Target X is out of bounds'")
    
    # Create value mapping from mask values to valid class indices
    unique_mask_values = sorted(list(sample_mask_values))
    
    print(f"\n🔧 SOLUTION: Map mask values to valid class indices [0, {NUM_CLASSES-1}]")
    
    # Check if we have the right number of unique values
    if len(unique_mask_values) == NUM_CLASSES:
        # Perfect match - create direct mapping
        value_mapping = {mask_val: class_idx for class_idx, mask_val in enumerate(unique_mask_values)}
        print(f"✅ Perfect match: {len(unique_mask_values)} mask values → {NUM_CLASSES} classes")
        
    elif len(unique_mask_values) > NUM_CLASSES:
        # More mask values than classes - use first NUM_CLASSES values
        print(f"⚠️ Found {len(unique_mask_values)} unique mask values but only {NUM_CLASSES} classes defined")
        print(f"   Using first {NUM_CLASSES} mask values: {unique_mask_values[:NUM_CLASSES]}")
        value_mapping = {unique_mask_values[i]: i for i in range(NUM_CLASSES)}
        
    else:
        # Fewer mask values than expected classes
        print(f"⚠️ Found {len(unique_mask_values)} unique mask values but {NUM_CLASSES} classes expected")
        print(f"   Adjusting NUM_CLASSES to match actual mask values")
        
        # Update NUM_CLASSES to match actual data, but preserve class name structure
        old_num_classes = NUM_CLASSES
        NUM_CLASSES = len(unique_mask_values)
        
        # Preserve original class names if available, otherwise create new ones
        if len(class_names) >= NUM_CLASSES:
            # Keep first NUM_CLASSES names
            class_names = class_names[:NUM_CLASSES]
        else:
            # Extend with generic names
            class_names = class_names + [f"class_{i}" for i in range(len(class_names), NUM_CLASSES)]
        
        # Create direct mapping
        value_mapping = {mask_val: class_idx for class_idx, mask_val in enumerate(unique_mask_values)}
        
        print(f"   Updated NUM_CLASSES: {old_num_classes} → {NUM_CLASSES}")
        print(f"   Preserved/updated class names: {class_names}")
        
        # IMPORTANT: Recreate model with correct number of classes
        print(f"\n🏗️ RECREATING MODEL with {NUM_CLASSES} classes...")
        model = model_class(
            encoder_name=BACKBONE,
            encoder_weights="imagenet",
            in_channels=3,
            classes=NUM_CLASSES,
        ).to(device)
        
        # Recreate optimizer and scheduler
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LEARNING_RATE,
            weight_decay=1e-4
        )
        
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=0.5,
            patience=5
        )
        
        print(f"✅ Model recreated with {NUM_CLASSES} classes")
    
    # Update class mappings to maintain consistency
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    idx_to_class = {idx: name for idx, name in enumerate(class_names)}
    
    print(f"\n📋 Final Value Mapping:")
    print(f"   Mask value → Class index: {value_mapping}")
    print(f"   Class index → Class name: {dict(list(idx_to_class.items())[:5])}{'...' if len(idx_to_class) > 5 else ''}")

else:
    print(f"✅ Mask values are already in correct range [0, {NUM_CLASSES-1}]")
    print(f"✅ No value mapping needed")

print(f"\n🎯 Final configuration maintained from labelmap:")
print(f"   Classes: {NUM_CLASSES}")
print(f"   Class names: {class_names}")
print(f"   Value range: [0, {NUM_CLASSES-1}]")
print(f"   Labelmap preserved: {'✅' if 'labelmap_found' in locals() and labelmap_found else '⚠️ Generated from masks'}")

### 🔄 Updated Dataset with Mask Value Mapping

In [ ]:
# 🔄 Updated Dataset Class with Value Mapping
class FixedSegmentationDataset(Dataset):
    def __init__(self, dataset, transforms=None, value_mapping=None):
        self.dataset = dataset
        self.transforms = transforms
        self.value_mapping = value_mapping
    
    def __len__(self):
        return len(self.dataset)
    
    def __getitem__(self, idx):
        item = self.dataset[idx]
        
        # Load image and mask
        image = cv2.imread(item['img_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(item['seg_map_path'], cv2.IMREAD_GRAYSCALE)
        
        # Apply value mapping if provided
        if self.value_mapping is not None:
            # Create a copy to avoid modifying original
            mapped_mask = mask.copy()
            for old_val, new_val in self.value_mapping.items():
                mapped_mask[mask == old_val] = new_val
            mask = mapped_mask
        
        # Apply transforms
        if self.transforms:
            transformed = self.transforms(image=image, mask=mask)
            image = transformed['image']
            mask = transformed['mask']
        
        return image, mask.long()

# Create value mapping if we found problematic values
if 'value_mapping' in locals():
    print(f"🔧 Creating datasets with value mapping: {value_mapping}")
    
    # Recreate datasets with mapping
    train_ds = FixedSegmentationDataset(train_dataset, transforms=train_transforms, value_mapping=value_mapping)
    val_ds = FixedSegmentationDataset(val_dataset, transforms=val_transforms, value_mapping=value_mapping)
    
else:
    print(f"✅ No value mapping needed, using original datasets")
    # Use original dataset class
    train_ds = SegmentationDataset(train_dataset, transforms=train_transforms)
    val_ds = SegmentationDataset(val_dataset, transforms=val_transforms)

# Recreate DataLoaders
train_loader = DataLoader(
    train_ds, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print(f"✅ Updated datasets created successfully!")
print(f"   Training batches: {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")

# 🧪 Test the fixed data loading
try:
    sample_image, sample_mask = next(iter(train_loader))
    print(f"📊 Fixed data loading test:")
    print(f"   Image shape: {sample_image.shape}")
    print(f"   Mask shape: {sample_mask.shape}")
    print(f"   Mask value range: [{sample_mask.min()}, {sample_mask.max()}]")
    print(f"   Unique mask values: {torch.unique(sample_mask)}")
    
    # Verify all values are in valid range
    max_mask_val = sample_mask.max().item()
    if max_mask_val >= NUM_CLASSES:
        print(f"❌ STILL PROBLEMATIC: Max mask value {max_mask_val} >= NUM_CLASSES {NUM_CLASSES}")
    else:
        print(f"✅ Mask values are now in valid range [0, {NUM_CLASSES-1}]")
        
except Exception as e:
    print(f"❌ Data loading test failed: {e}")

### 🐛 CUDA Error Debug & Validation

In [ ]:
# 🐛 COMPREHENSIVE CUDA ERROR DEBUG
# CUDA "device-side assert triggered" usually means invalid class indices

print("🔍 THOROUGH VALIDATION - Finding CUDA Error Source")
print("="*60)

# Enable synchronous CUDA operations for better error reporting
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# Test 1: Validate ALL mask files (not just first 10)
print("📋 Test 1: Validating ALL mask files...")
all_mask_values = set()
invalid_files = []

for i, mask_file in enumerate(mask_files):
    mask_path = os.path.join(LOCAL_DOWNLOAD_MASK_DIR, mask_file)
    try:
        mask = np.array(Image.open(mask_path))
        unique_vals = np.unique(mask)
        all_mask_values.update(unique_vals)
        
        # Check for any values >= NUM_CLASSES
        if np.max(unique_vals) >= NUM_CLASSES:
            invalid_files.append((mask_file, unique_vals, np.max(unique_vals)))
            
    except Exception as e:
        print(f"  ❌ Error reading {mask_file}: {e}")

print(f"   All unique values across ALL masks: {sorted(all_mask_values)}")
print(f"   NUM_CLASSES: {NUM_CLASSES} (valid range: [0, {NUM_CLASSES-1}])")
print(f"   Invalid files found: {len(invalid_files)}")

if invalid_files:
    print(f"\n❌ FOUND THE PROBLEM! Files with invalid values:")
    for file, vals, max_val in invalid_files[:5]:  # Show first 5
        print(f"   {file}: max value {max_val} >= {NUM_CLASSES}")
    
    # Create COMPLETE value mapping for ALL found values
    print(f"\n🔧 Creating COMPLETE value mapping...")
    all_unique_values = sorted(list(all_mask_values))
    
    if len(all_unique_values) > NUM_CLASSES:
        print(f"⚠️ Found {len(all_unique_values)} unique values but only {NUM_CLASSES} classes")
        print(f"   Will map first {NUM_CLASSES} values to valid range")
        # Map all values, but excess ones will map to last class
        complete_value_mapping = {}
        for i, val in enumerate(all_unique_values):
            if i < NUM_CLASSES:
                complete_value_mapping[val] = i
            else:
                complete_value_mapping[val] = NUM_CLASSES - 1  # Map to last class
    else:
        complete_value_mapping = {val: i for i, val in enumerate(all_unique_values)}
    
    print(f"   Complete mapping: {complete_value_mapping}")
    
    # Update the global value_mapping
    value_mapping = complete_value_mapping

# Test 2: Validate data loader with complete mapping
print(f"\n📋 Test 2: Testing data loader with complete mapping...")

# Recreate datasets with COMPLETE mapping
if 'value_mapping' in locals():
    print(f"🔧 Using complete value mapping: {len(value_mapping)} mappings")
    train_ds = FixedSegmentationDataset(train_dataset, transforms=train_transforms, value_mapping=value_mapping)
    val_ds = FixedSegmentationDataset(val_dataset, transforms=val_transforms, value_mapping=value_mapping)
else:
    print(f"✅ Using original datasets (no mapping needed)")
    train_ds = SegmentationDataset(train_dataset, transforms=train_transforms)
    val_ds = SegmentationDataset(val_dataset, transforms=val_transforms)

# Recreate data loaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)  # Reduced workers for debugging
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=False)

# Test 3: Comprehensive batch validation
print(f"\n📋 Test 3: Validating ENTIRE first batch...")
try:
    sample_images, sample_masks = next(iter(train_loader))
    print(f"   Batch image shape: {sample_images.shape}")
    print(f"   Batch mask shape: {sample_masks.shape}")
    
    # Check EVERY mask in the batch
    batch_size = sample_masks.shape[0]
    for i in range(batch_size):
        mask = sample_masks[i]
        unique_vals = torch.unique(mask)
        min_val, max_val = mask.min().item(), mask.max().item()
        
        print(f"   Sample {i}: range [{min_val}, {max_val}], unique: {unique_vals.tolist()}")
        
        if max_val >= NUM_CLASSES:
            print(f"   ❌ PROBLEM: Sample {i} has value {max_val} >= {NUM_CLASSES}")
        elif min_val < 0:
            print(f"   ❌ PROBLEM: Sample {i} has negative value {min_val}")
            
    print(f"✅ Batch validation complete!")
    
except Exception as e:
    print(f"❌ Batch validation failed: {e}")

# Test 4: Small forward pass test
print(f"\n📋 Test 4: Testing model forward pass with valid data...")
try:
    model.eval()
    with torch.no_grad():
        # Use a small batch for testing
        test_images = sample_images[:2].to(device)  # Just 2 samples
        test_masks = sample_masks[:2].to(device)
        
        print(f"   Test input shape: {test_images.shape}")
        print(f"   Test target shape: {test_masks.shape}")
        print(f"   Test target range: [{test_masks.min()}, {test_masks.max()}]")
        
        # Forward pass
        outputs = model(test_images)
        print(f"   Model output shape: {outputs.shape}")
        
        # Test loss calculation
        loss = criterion(outputs, test_masks)
        print(f"   Loss calculation: {loss.item():.4f}")
        
    model.train()
    print(f"✅ Forward pass test successful!")
    
except Exception as e:
    print(f"❌ Forward pass test failed: {e}")
    print(f"   This is likely the source of your CUDA error!")

print(f"\n🎯 VALIDATION SUMMARY:")
print(f"   Total mask files checked: {len(mask_files)}")
print(f"   Unique values found: {sorted(all_mask_values)}")
print(f"   NUM_CLASSES: {NUM_CLASSES}")
print(f"   Value mapping applied: {'Yes' if 'value_mapping' in locals() else 'No'}")
print(f"   Ready for training: {'✅' if len(invalid_files) == 0 else '⚠️ Check above errors'}")
print("="*60)

### 🎨 RGB Labelmap to Grayscale Mapping Fix

In [ ]:
# 🎨 CORRECT SOLUTION: Parse RGB Labelmap and Map to Grayscale Values
print("🎨 RGB LABELMAP TO GRAYSCALE MAPPING")
print("="*60)

# Parse the RGB labelmap.txt file
rgb_classes = {}
grayscale_to_rgb = {}

if os.path.exists(local_labelmap_path):
    print(f"📖 Reading RGB labelmap from: {local_labelmap_path}")
    
    with open(local_labelmap_path, 'r') as f:
        lines = f.readlines()
    
    print(f"📋 Found {len(lines)} lines in labelmap:")
    
    for i, line in enumerate(lines):
        line = line.strip()
        if line and ':' in line:
            # Parse format: "ClassName:R,G,B"
            class_name, rgb_str = line.split(':', 1)
            class_name = class_name.strip()
            
            try:
                # Parse RGB values
                r, g, b = map(int, rgb_str.split(','))
                rgb_classes[class_name] = (r, g, b)
                
                # Convert RGB to grayscale using standard formula
                # Grayscale = 0.299*R + 0.587*G + 0.114*B
                grayscale_value = int(0.299 * r + 0.587 * g + 0.114 * b)
                grayscale_to_rgb[grayscale_value] = (class_name, i, (r, g, b))
                
                print(f"   {i:2d}. {class_name:15s}: RGB({r:3d},{g:3d},{b:3d}) → Grayscale({grayscale_value:3d})")
                
            except ValueError as e:
                print(f"   ❌ Error parsing line {i}: '{line}' - {e}")
    
    print(f"\n📊 Parsed {len(rgb_classes)} RGB classes")
    
else:
    print(f"❌ Labelmap file not found: {local_labelmap_path}")
    print("Cannot create RGB mapping without labelmap file!")

# Now analyze actual mask values and find the correct mapping
print(f"\n🔍 ANALYZING ACTUAL MASK VALUES...")

# Get unique values from masks (we found these in debug: 0, 11, 113, 118, 125, 139, 159, 227)
actual_mask_values = sorted(list(all_mask_values)) if 'all_mask_values' in locals() else []

if not actual_mask_values:
    # Re-scan if not available
    print("Re-scanning mask files for unique values...")
    actual_mask_values = set()
    for mask_file in mask_files[:5]:  # Sample first 5
        mask_path = os.path.join(LOCAL_DOWNLOAD_MASK_DIR, mask_file)
        try:
            mask = np.array(Image.open(mask_path))
            actual_mask_values.update(np.unique(mask))
        except:
            pass
    actual_mask_values = sorted(list(actual_mask_values))

print(f"📋 Actual mask values found: {actual_mask_values}")
print(f"📋 Expected grayscale values from RGB: {sorted(grayscale_to_rgb.keys())}")

# Create the mapping strategy
print(f"\n🔧 CREATING GRAYSCALE TO CLASS MAPPING...")

if len(grayscale_to_rgb) > 0:
    # Strategy 1: Try to match actual mask values to expected grayscale values
    grayscale_mapping = {}
    
    # Direct matches first
    for mask_val in actual_mask_values:
        if mask_val in grayscale_to_rgb:
            class_name, class_idx, rgb = grayscale_to_rgb[mask_val]
            grayscale_mapping[mask_val] = class_idx
            print(f"   Direct match: {mask_val} → Class {class_idx} ({class_name})")
    
    # For unmatched values, find closest grayscale values
    expected_grayscales = list(grayscale_to_rgb.keys())
    
    for mask_val in actual_mask_values:
        if mask_val not in grayscale_mapping:
            # Find closest expected grayscale value
            closest_grayscale = min(expected_grayscales, key=lambda x: abs(x - mask_val))
            class_name, class_idx, rgb = grayscale_to_rgb[closest_grayscale]
            grayscale_mapping[mask_val] = class_idx
            print(f"   Closest match: {mask_val} → {closest_grayscale} → Class {class_idx} ({class_name})")
    
    # Update global variables
    value_mapping = grayscale_mapping
    NUM_CLASSES = len(rgb_classes)
    class_names = list(rgb_classes.keys())
    
    print(f"\n✅ FINAL MAPPING CREATED:")
    print(f"   Mask value → Class index: {value_mapping}")
    print(f"   Classes: {NUM_CLASSES}")
    print(f"   Class names: {class_names}")
    
    # Update class mappings
    class_to_idx = {name: idx for idx, name in enumerate(class_names)}
    idx_to_class = {idx: name for idx, name in enumerate(class_names)}
    
    # RECREATE MODEL with correct number of classes
    print(f"\n🏗️ RECREATING MODEL with {NUM_CLASSES} classes from labelmap...")
    model = model_class(
        encoder_name=BACKBONE,
        encoder_weights="imagenet",
        in_channels=3,
        classes=NUM_CLASSES,
    ).to(device)
    
    # Recreate optimizer and scheduler
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=1e-4
    )
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=5
    )
    
    print(f"✅ Model recreated with {NUM_CLASSES} classes from RGB labelmap")
    
else:
    print(f"⚠️ No RGB labelmap found, using fallback mapping")
    # Fallback: direct mapping of mask values to sequential indices  
    value_mapping = {val: idx for idx, val in enumerate(actual_mask_values[:14])}  # Limit to 14 classes
    NUM_CLASSES = min(len(actual_mask_values), 14)

print(f"\n🎯 FINAL CONFIGURATION:")
print(f"   RGB classes from labelmap: {len(rgb_classes)}")
print(f"   Actual mask values: {len(actual_mask_values)}")
print(f"   Final NUM_CLASSES: {NUM_CLASSES}")
print(f"   Value mapping: {dict(list(value_mapping.items())[:5])}{'...' if len(value_mapping) > 5 else ''}")
print("="*60)

### 4.1: 📊 Training Metrics and Utilities

In [ ]:
# 📊 Training Utilities and Metrics
from tqdm import tqdm
import time

def calculate_iou(pred_mask, true_mask, num_classes):
    """Calculate Intersection over Union (IoU) for each class"""
    ious = []
    pred_mask = pred_mask.flatten()
    true_mask = true_mask.flatten()
    
    for cls in range(num_classes):
        pred_inds = pred_mask == cls
        target_inds = true_mask == cls
        intersection = (pred_inds & target_inds).sum().float()
        union = (pred_inds | target_inds).sum().float()
        
        if union == 0:
            ious.append(float('nan'))  # No ground truth or prediction for this class
        else:
            ious.append((intersection / union).cpu().item())
    
    return ious

def train_one_epoch(model, train_loader, criterion, optimizer, device, epoch):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    running_iou = 0.0
    
    progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    
    for batch_idx, (images, masks) in enumerate(progress_bar):
        images = images.to(device)
        masks = masks.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Calculate metrics
        running_loss += loss.item()
        
        # Calculate IoU
        with torch.no_grad():
            pred_masks = torch.argmax(outputs, dim=1)
            batch_ious = []
            for i in range(images.size(0)):
                ious = calculate_iou(pred_masks[i], masks[i], NUM_CLASSES)
                # Take mean of valid IoUs (ignore nan values)
                valid_ious = [iou for iou in ious if not np.isnan(iou)]
                if valid_ious:
                    batch_ious.append(np.mean(valid_ious))
            
            if batch_ious:
                running_iou += np.mean(batch_ious)
        
        # Update progress bar
        progress_bar.set_postfix({
            'Loss': f'{loss.item():.4f}',
            'Avg Loss': f'{running_loss/(batch_idx+1):.4f}',
            'IoU': f'{running_iou/(batch_idx+1):.4f}'
        })
    
    epoch_loss = running_loss / len(train_loader)
    epoch_iou = running_iou / len(train_loader)
    
    return epoch_loss, epoch_iou

def validate_one_epoch(model, val_loader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    running_loss = 0.0
    running_iou = 0.0
    
    with torch.no_grad():
        progress_bar = tqdm(val_loader, desc='Validation')
        
        for batch_idx, (images, masks) in enumerate(progress_bar):
            images = images.to(device)
            masks = masks.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            running_loss += loss.item()
            
            # Calculate IoU
            pred_masks = torch.argmax(outputs, dim=1)
            batch_ious = []
            for i in range(images.size(0)):
                ious = calculate_iou(pred_masks[i], masks[i], NUM_CLASSES)
                valid_ious = [iou for iou in ious if not np.isnan(iou)]
                if valid_ious:
                    batch_ious.append(np.mean(valid_ious))
            
            if batch_ious:
                running_iou += np.mean(batch_ious)
            
            progress_bar.set_postfix({
                'Loss': f'{loss.item():.4f}',
                'Avg Loss': f'{running_loss/(batch_idx+1):.4f}',
                'IoU': f'{running_iou/(batch_idx+1):.4f}'
            })
    
    epoch_loss = running_loss / len(val_loader)
    epoch_iou = running_iou / len(val_loader)
    
    return epoch_loss, epoch_iou

print("✅ Training utilities loaded!")
print(f"   Ready to train {MODEL_ARCH} for {NUM_EPOCHS} epochs")
print(f"   Training batches: {len(train_loader)}")
print(f"   Validation batches: {len(val_loader)}")

### 4.2: 🏃‍♂️ Main Training Loop

In [ ]:
# 🏃‍♂️ ACTUAL TRAINING LOOP - This creates the TRAINED model!
print("🚀 Starting training...")
print(f"   Model: {MODEL_ARCH} with {BACKBONE} backbone")
print(f"   Dataset: {len(train_dataset)} train, {len(val_dataset)} val samples")
print(f"   Training for {NUM_EPOCHS} epochs")
print("="*60)

# Training history
train_losses = []
train_ious = []
val_losses = []
val_ious = []
best_val_iou = 0.0

# Start training
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    print(f"\n📅 Epoch {epoch+1}/{NUM_EPOCHS}")
    
    # Train
    train_loss, train_iou = train_one_epoch(
        model, train_loader, criterion, optimizer, device, epoch
    )
    
    # Validate
    val_loss, val_iou = validate_one_epoch(
        model, val_loader, criterion, device
    )
    
    # Update learning rate
    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save metrics
    train_losses.append(train_loss)
    train_ious.append(train_iou)
    val_losses.append(val_loss)
    val_ious.append(val_iou)
    
    # Print epoch summary
    print(f"📊 Epoch {epoch+1} Results:")
    print(f"   Train Loss: {train_loss:.4f} | Train IoU: {train_iou:.4f}")
    print(f"   Val Loss: {val_loss:.4f} | Val IoU: {val_iou:.4f}")
    print(f"   Learning Rate: {current_lr:.2e}")
    
    # Save best model
    if val_iou > best_val_iou:
        best_val_iou = val_iou
        best_model_path = f"{MODEL_SAVE_DIR}/best_{MODEL_ARCH}_{BACKBONE}_epoch_{epoch+1}.pth"
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'train_loss': train_loss,
            'val_loss': val_loss,
            'val_iou': val_iou,
            'model_arch': MODEL_ARCH,
            'backbone': BACKBONE,
            'num_classes': NUM_CLASSES,
            'class_names': class_names
        }, best_model_path)
        print(f"💾 New best model saved! IoU: {val_iou:.4f}")

# Training completed
end_time = time.time()
training_time = end_time - start_time

print("="*60)
print("🎉 TRAINING COMPLETED!")
print(f"⏱️ Total training time: {training_time/3600:.2f} hours")
print(f"🏆 Best validation IoU: {best_val_iou:.4f}")
print(f"💾 Best model saved to: {best_model_path}")
print(f"📊 Training history: {len(train_losses)} epochs")

# Final model save
final_model_path = f"{MODEL_SAVE_DIR}/final_{MODEL_ARCH}_{BACKBONE}.pth"
torch.save({
    'model_state_dict': model.state_dict(),
    'training_history': {
        'train_losses': train_losses,
        'train_ious': train_ious,
        'val_losses': val_losses,
        'val_ious': val_ious
    },
    'model_arch': MODEL_ARCH,
    'backbone': BACKBONE,
    'num_classes': NUM_CLASSES,
    'class_names': class_names
}, final_model_path)

print(f"💾 Final model saved to: {final_model_path}")
print("\n🎯 YOU NOW HAVE A TRAINED MODEL! 🎯")

### 4.3: 📈 Training Results Visualization

In [ ]:
# 📈 Plot Training Results
import matplotlib.pyplot as plt

# Create training plots
fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))

# Loss plots
epochs = range(1, len(train_losses) + 1)

ax1.plot(epochs, train_losses, 'b-', label='Training Loss', linewidth=2)
ax1.plot(epochs, val_losses, 'r-', label='Validation Loss', linewidth=2)
ax1.set_title('Training vs Validation Loss')
ax1.set_xlabel('Epochs')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True)

# IoU plots
ax2.plot(epochs, train_ious, 'b-', label='Training IoU', linewidth=2)
ax2.plot(epochs, val_ious, 'r-', label='Validation IoU', linewidth=2)
ax2.set_title('Training vs Validation IoU')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('IoU')
ax2.legend()
ax2.grid(True)

# Learning rate plot
ax3.plot(epochs, [optimizer.param_groups[0]['lr']] * len(epochs), 'g-', linewidth=2)
ax3.set_title('Learning Rate Schedule')
ax3.set_xlabel('Epochs')
ax3.set_ylabel('Learning Rate')
ax3.set_yscale('log')
ax3.grid(True)

# Best metrics summary
metrics_text = f"""
Final Training Results:
━━━━━━━━━━━━━━━━━━━━━━

🏆 Best Validation IoU: {best_val_iou:.4f}
📉 Final Training Loss: {train_losses[-1]:.4f}
📉 Final Validation Loss: {val_losses[-1]:.4f}
📊 Final Training IoU: {train_ious[-1]:.4f}
📊 Final Validation IoU: {val_ious[-1]:.4f}

🔧 Model: {MODEL_ARCH}
🔧 Backbone: {BACKBONE}
🔧 Classes: {NUM_CLASSES}
⏱️ Training Time: {training_time/3600:.2f} hours
"""

ax4.text(0.1, 0.5, metrics_text, transform=ax4.transAxes, fontsize=12,
         verticalalignment='center', fontfamily='monospace')
ax4.set_xlim(0, 1)
ax4.set_ylim(0, 1)
ax4.axis('off')

plt.tight_layout()
plt.savefig(f'{LOGS_DIR}/training_results.png', dpi=300, bbox_inches='tight')
plt.show()

print("✅ Training visualization complete!")
print(f"📊 Plot saved to: {LOGS_DIR}/training_results.png")

# Print final summary
print("\n" + "="*60)
print("🎯 TRAINING SUMMARY")
print("="*60)
print(f"✅ Status: TRAINED MODEL READY!")
print(f"🏗️ Architecture: {MODEL_ARCH} with {BACKBONE}")
print(f"📊 Classes: {NUM_CLASSES}")
print(f"🏆 Best IoU: {best_val_iou:.4f}")
print(f"💾 Best model: {best_model_path}")
print(f"💾 Final model: {final_model_path}")
print("="*60)